# V70 - Label clipping

Pre-registered plan: 71% of labels are zero and the nonzero tail includes micro-relevance that may be annotation noise. Train word-only Ridge on labels clipped at 0.05 and compare against unclipped training under identical folds and a fresh seed. One threshold, fixed up front, so there is no sweep to overfit. Same locks.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold

SEED = 212
N_SPLITS = 5
GATE_FLOOR = 0.002
TAU = 0.05
rng = np.random.RandomState(SEED)

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v70")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve())

## Clipped vs raw training, identical folds

Evaluation always uses the original graded labels. Only the training targets differ.

In [ ]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)
print(f"fraction below {TAU} (nonzero): {((Y > 0) & (Y < TAU)).mean():.4f}")
YC = np.where(Y < TAU, 0.0, Y)
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in list(train["user_id"])]
W = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(docs).toarray()

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

def ridge_oof(T, alpha=1.0):
    oof = np.zeros_like(Y)
    fids = np.zeros(n_train, int)
    skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    for fi, (tr, va) in enumerate(skf.split(W, strata, fold_groups)):
        for j in range(len(MODS)):
            m = Ridge(alpha=alpha)
            m.fit(W[tr], T[tr, j])
            oof[va, j] = m.predict(W[va])
        fids[va] = fi + 1
    return oof, fids

oof_raw, fid = ridge_oof(Y)
oof_cli, _ = ridge_oof(YC)
base = ndcg5(Y, oof_raw)
d = ndcg5(Y, oof_cli) - base
lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
wf = min(ndcg5(Y[fid == k], oof_cli[fid == k]).mean() - base[fid == k].mean() for k in range(1, N_SPLITS + 1))
print(f"raw={base.mean():.5f} clipped={ndcg5(Y, oof_cli).mean():.5f} delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}] worst-fold={wf:+.5f}")
print("ADOPT clipping" if (lo > GATE_FLOOR and wf >= 0) else "KEEP raw labels; micro-relevance carries signal")